# Enrichissement de `previous_application`

## Objectif

Ce notebook enrichira progressivement `previous_application.csv` avec les trois historiques liés par `SK_ID_PREV`. Cette première version traite uniquement `POS_CASH_balance.csv`, conformément à l'analyse et au choix de variables réalisés avant l'implémentation.

`POS_CASH_balance` contient plusieurs observations mensuelles pour une même ancienne demande. Elle doit donc être agrégée pour obtenir une ligne par `SK_ID_PREV` avant la jointure. La jointure gauche doit conserver les 1 670 214 anciennes demandes de `previous_application`. `installments_payments` et `credit_card_balance` seront étudiées séparément avant leur ajout.

In [1]:
from pathlib import Path

import pandas as pd

# ---------- Chemins des données ----------
RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
PREVIOUS_APPLICATION_PATH = RAW_DIR / "previous_application.csv"
POS_CASH_PATH = RAW_DIR / "POS_CASH_balance.csv"
PREVIOUS_ENRICHI_PATH = PROCESSED_DIR / "previous_application_enrichi.csv"

for data_path in [PREVIOUS_APPLICATION_PATH, POS_CASH_PATH]:
    if not data_path.is_file():
        raise FileNotFoundError(f"Fichier introuvable : {data_path.resolve()}")

## Chargement et contrôle initial

Des types compacts sont utilisés pour limiter la mémoire nécessaire au chargement des dix millions de lignes POS/CASH.

In [2]:
# ---------- Chargement des données ----------
previous_application = pd.read_csv(PREVIOUS_APPLICATION_PATH)
pos_cash = pd.read_csv(
    POS_CASH_PATH,
    dtype={
        "SK_ID_PREV": "int32",
        "SK_ID_CURR": "int32",
        "MONTHS_BALANCE": "int8",
        "SK_DPD": "int16",
        "SK_DPD_DEF": "int16",
        "NAME_CONTRACT_STATUS": "category",
    },
)

print(f"previous_application : {previous_application.shape}")
print(f"POS_CASH_balance : {pos_cash.shape}")
display(pos_cash.head())

previous_application : (1670214, 37)
POS_CASH_balance : (10001358, 8)


,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,CNT_INSTALMENT,CNT_INSTALMENT_FUTURE,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,1803195,182943,-31,48.0,45.0,Active,0,0
1,1715348,367990,-33,36.0,35.0,Active,0,0
2,1784872,397406,-32,12.0,9.0,Active,0,0
3,1903291,269225,-35,48.0,42.0,Active,0,0
4,2341044,334279,-35,36.0,35.0,Active,0,0


In [3]:
# ---------- Validation des données ----------
PREVIOUS_ID = "SK_ID_PREV"
MONTH_COLUMN = "MONTHS_BALANCE"
required_previous_columns = {PREVIOUS_ID, "SK_ID_CURR"}
required_pos_columns = {
    PREVIOUS_ID, "SK_ID_CURR", MONTH_COLUMN,
    "CNT_INSTALMENT_FUTURE", "SK_DPD_DEF",
}

if not required_previous_columns.issubset(previous_application.columns):
    missing = required_previous_columns.difference(previous_application.columns)
    raise KeyError(f"Colonnes absentes de previous_application : {sorted(missing)}")
if not required_pos_columns.issubset(pos_cash.columns):
    missing = required_pos_columns.difference(pos_cash.columns)
    raise KeyError(f"Colonnes absentes de POS_CASH_balance : {sorted(missing)}")

assert previous_application[PREVIOUS_ID].notna().all()
assert previous_application[PREVIOUS_ID].is_unique
assert pos_cash[[PREVIOUS_ID, "SK_ID_CURR", MONTH_COLUMN]].notna().all().all()
assert not pos_cash.duplicated([PREVIOUS_ID, MONTH_COLUMN]).any()
assert pos_cash.groupby(PREVIOUS_ID)["SK_ID_CURR"].nunique().le(1).all()
print("Validation des identifiants réussie.")

Validation des identifiants réussie.


## Logique de jointure et feature engineering entre `POS_CASH` et `previous_application`

### Colonnes transformées

- `MONTHS_BALANCE` produit `POS_MONTH_COUNT` et permet de sélectionner le mois le plus récent ;
- `CNT_INSTALMENT_FUTURE` produit `POS_REMAINING_INSTALLMENTS_LATEST` ;
- `SK_DPD_DEF` produit `POS_EVER_DPD` et `POS_EVER_SEVERE_DPD`.

`SK_DPD_DEF` est préféré à `SK_DPD`, car il ignore les petits impayés tolérés. Un retard sévère correspond ici à au moins 61 jours.

### Colonnes non retenues

- `SK_ID_CURR` est redondante avec celle de la table parente et sert uniquement au contrôle de cohérence ;
- `CNT_INSTALMENT` est susceptible d'évoluer et apporte une information proche des échéances restantes ;
- `NAME_CONTRACT_STATUS` possède plusieurs catégories mensuelles non prioritaires ;
- `SK_DPD` est remplacée par sa version avec tolérance.

Les colonnes brutes ne sont jamais supprimées de leur fichier source.

In [4]:
# ---------- Feature engineering ----------
pos_cash = pos_cash.assign(
    _IS_DPD=pos_cash["SK_DPD_DEF"].gt(0).astype("int8"),
    _IS_SEVERE_DPD=pos_cash["SK_DPD_DEF"].ge(61).astype("int8"),
)

latest_row_index = pos_cash.groupby(PREVIOUS_ID)[MONTH_COLUMN].idxmax()
latest_installments = (
    pos_cash.loc[latest_row_index, [PREVIOUS_ID, "CNT_INSTALMENT_FUTURE"]]
    .set_index(PREVIOUS_ID)
    .rename(columns={
        "CNT_INSTALMENT_FUTURE": "POS_REMAINING_INSTALLMENTS_LATEST"
    })
)

pos_cash_agg = (
    pos_cash.groupby(PREVIOUS_ID, observed=True)
    .agg(
        POS_MONTH_COUNT=(MONTH_COLUMN, "size"),
        POS_EVER_DPD=("_IS_DPD", "max"),
        POS_EVER_SEVERE_DPD=("_IS_SEVERE_DPD", "max"),
    )
    .join(latest_installments, validate="one_to_one")
    .reset_index()
)

assert pos_cash_agg[PREVIOUS_ID].is_unique
del pos_cash
print(f"POS_CASH agrégé : {pos_cash_agg.shape}")
display(pos_cash_agg.head())

POS_CASH agrégé : (936325, 5)


,SK_ID_PREV,POS_MONTH_COUNT,POS_EVER_DPD,POS_EVER_SEVERE_DPD,POS_REMAINING_INSTALLMENTS_LATEST
0,1000001,3,0,0,0.0
1,1000002,5,0,0,0.0
2,1000003,4,0,0,9.0
3,1000004,8,0,0,0.0
4,1000005,11,0,0,0.0


In [5]:
# ---------- Couverture des clés ----------
previous_ids = pd.Index(previous_application[PREVIOUS_ID])
pos_ids = pd.Index(pos_cash_agg[PREVIOUS_ID])
orphan_pos_ids = pos_ids.difference(previous_ids)
previous_without_pos_ids = previous_ids.difference(pos_ids)

print(f"Identifiants POS absents de previous_application : {len(orphan_pos_ids):,}")
print(f"Anciennes demandes sans historique POS : {len(previous_without_pos_ids):,}")

Identifiants POS absents de previous_application : 37,422
Anciennes demandes sans historique POS : 771,311


## réalisation de la jointure entre `POS_CASH` et `previous_application`

La jointure gauche conserve toutes les anciennes demandes. Les indicateurs de retard restent manquants en l'absence d'historique, tandis que `POS_HAS_HISTORY` permet d'identifier explicitement cette situation.

In [6]:
# ---------- Jointure des données ----------
previous_application_enrichi = previous_application.merge(
    pos_cash_agg, on=PREVIOUS_ID, how="left", validate="one_to_one"
)
previous_application_enrichi["POS_HAS_HISTORY"] = (
    previous_application_enrichi["POS_MONTH_COUNT"].notna().astype("int8")
)
previous_application_enrichi["POS_MONTH_COUNT"] = (
    previous_application_enrichi["POS_MONTH_COUNT"].fillna(0).astype("int16")
)

assert len(previous_application_enrichi) == len(previous_application)
assert previous_application_enrichi[PREVIOUS_ID].is_unique

In [7]:
# ---------- Contrôle avant et après la jointure ----------
controle_jointure = pd.DataFrame({
    "étape": ["avant", "après"],
    "nombre de lignes": [len(previous_application), len(previous_application_enrichi)],
    "nombre de colonnes": [
        previous_application.shape[1], previous_application_enrichi.shape[1]
    ],
})
display(controle_jointure)
print("Toutes les validations sont réussies.")

,étape,nombre de lignes,nombre de colonnes
0,avant,1670214,37
1,après,1670214,42


Toutes les validations sont réussies.


In [8]:
# ---------- Aperçu après l'étape POS/CASH ----------
display(previous_application_enrichi.head())

,SK_ID_PREV,SK_ID_CURR,NAME_CONTRACT_TYPE,AMT_ANNUITY,AMT_APPLICATION,AMT_CREDIT,AMT_DOWN_PAYMENT,AMT_GOODS_PRICE,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,...,DAYS_FIRST_DUE,DAYS_LAST_DUE_1ST_VERSION,DAYS_LAST_DUE,DAYS_TERMINATION,NFLAG_INSURED_ON_APPROVAL,POS_MONTH_COUNT,POS_EVER_DPD,POS_EVER_SEVERE_DPD,POS_REMAINING_INSTALLMENTS_LATEST,POS_HAS_HISTORY
0,2030495,271877,Consumer loans,1730.430,17145.0,17145.0,0.0,17145.0,SATURDAY,15,...,-42.0,300.0,-42.0,-37.0,0.0,2,0.0,0.0,0.0,1
1,2802425,108129,Cash loans,25188.615,607500.0,679671.0,NaN,607500.0,THURSDAY,11,...,-134.0,916.0,365243.0,365243.0,1.0,5,0.0,0.0,32.0,1
2,2523466,122040,Cash loans,15060.735,112500.0,136444.5,NaN,112500.0,TUESDAY,11,...,-271.0,59.0,365243.0,365243.0,1.0,10,0.0,0.0,3.0,1
3,2819243,176158,Cash loans,47041.335,450000.0,470790.0,NaN,450000.0,MONDAY,7,...,-482.0,-152.0,-182.0,-177.0,1.0,12,0.0,0.0,0.0,1
4,1784265,202054,Cash loans,31924.395,337500.0,404055.0,NaN,337500.0,THURSDAY,9,...,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,0


## Logique de jointure et feature engineering entre `installments_payments` et `previous_application`


Une échéance peut être répartie sur plusieurs lignes de paiement. Le traitement comporte donc deux niveaux : consolidation par échéance, puis agrégation par `SK_ID_PREV`. Cette méthode évite de compter plusieurs fois le montant attendu.

### Traitement retenu

La clé logique d'une échéance est composée de `SK_ID_PREV`, `NUM_INSTALMENT_NUMBER` et `NUM_INSTALMENT_VERSION`. Pour chaque échéance, la date et le montant attendus sont uniques, les paiements partiels sont additionnés et leur date la plus tardive est retenue.

Les variables finales sont : nombre d'échéances, proportion de retards, retard maximal, proportion de sous-paiements et rapport entre total payé et total attendu. `SK_ID_CURR` sert uniquement au contrôle de cohérence, car la jointure utilise `SK_ID_PREV`. Les paiements sans date ou montant réel sont exclus des ratios concernés au lieu d'être assimilés à zéro.

In [9]:
# ---------- Chargement des paiements ----------
INSTALLMENTS_PATH = RAW_DIR / "installments_payments.csv"
if not INSTALLMENTS_PATH.is_file():
    raise FileNotFoundError(f"Fichier introuvable : {INSTALLMENTS_PATH.resolve()}")

installments = pd.read_csv(
    INSTALLMENTS_PATH,
    dtype={
        "SK_ID_PREV": "int32",
        "SK_ID_CURR": "int32",
        "NUM_INSTALMENT_VERSION": "float32",
        "NUM_INSTALMENT_NUMBER": "int16",
        "DAYS_INSTALMENT": "float32",
        "DAYS_ENTRY_PAYMENT": "float32",
        "AMT_INSTALMENT": "float32",
        "AMT_PAYMENT": "float32",
    },
)
print(f"installments_payments : {installments.shape}")
display(installments.head())

installments_payments : (13605401, 8)


,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT
0,1054186,161674,1.0,6,-1180.0,-1187.0,6948.359863,6948.359863
1,1330831,151639,0.0,34,-2156.0,-2156.0,1716.525024,1716.525024
2,2085231,193053,2.0,1,-63.0,-63.0,25425.000000,25425.000000
3,2452527,199697,1.0,3,-2418.0,-2426.0,24350.130859,24350.130859
4,2714724,167756,1.0,2,-1383.0,-1366.0,2165.040039,2160.584961


In [10]:
# ---------- Validation des paiements ----------
INSTALLMENT_KEYS = [
    PREVIOUS_ID, "NUM_INSTALMENT_NUMBER", "NUM_INSTALMENT_VERSION"
]
required_installment_columns = set(INSTALLMENT_KEYS).union({
    "SK_ID_CURR", "DAYS_INSTALMENT", "DAYS_ENTRY_PAYMENT",
    "AMT_INSTALMENT", "AMT_PAYMENT",
})
if not required_installment_columns.issubset(installments.columns):
    missing = required_installment_columns.difference(installments.columns)
    raise KeyError(f"Colonnes absentes de installments_payments : {sorted(missing)}")

assert installments[INSTALLMENT_KEYS + ["SK_ID_CURR"]].notna().all().all()
assert installments.groupby(PREVIOUS_ID)["SK_ID_CURR"].nunique().le(1).all()
print(
    "Lignes supplémentaires partageant une clé d'échéance : "
    f"{installments.duplicated(INSTALLMENT_KEYS).sum():,}"
)

Lignes supplémentaires partageant une clé d'échéance : 653,483


In [11]:
# ---------- Consolidation par échéance ----------
installments["_HAS_PAYMENT"] = installments["AMT_PAYMENT"].notna().astype("int8")

installment_level = (
    installments.groupby(INSTALLMENT_KEYS, as_index=False, observed=True)
    .agg(
        INSTALLMENT_DUE_DATE=("DAYS_INSTALMENT", "first"),
        INSTALLMENT_LAST_PAYMENT_DATE=("DAYS_ENTRY_PAYMENT", "max"),
        INSTALLMENT_EXPECTED_AMOUNT=("AMT_INSTALMENT", "first"),
        INSTALLMENT_PAID_AMOUNT=("AMT_PAYMENT", "sum"),
        _HAS_PAYMENT=("_HAS_PAYMENT", "max"),
    )
)
installment_level["INSTALLMENT_PAID_AMOUNT"] = (
    installment_level["INSTALLMENT_PAID_AMOUNT"]
    .where(installment_level["_HAS_PAYMENT"].eq(1))
)

assert not installment_level.duplicated(INSTALLMENT_KEYS).any()
del installments
print(f"Échéances consolidées : {installment_level.shape[0]:,}")

Échéances consolidées : 12,951,918


In [12]:
# ---------- Feature engineering des échéances ----------
observed_date = installment_level["INSTALLMENT_LAST_PAYMENT_DATE"].notna()
observed_amount = installment_level["INSTALLMENT_PAID_AMOUNT"].notna()
days_late = (
    installment_level["INSTALLMENT_LAST_PAYMENT_DATE"]
    - installment_level["INSTALLMENT_DUE_DATE"]
).clip(lower=0)

installment_level["_DAYS_LATE"] = days_late.where(observed_date)
installment_level["_IS_LATE"] = (
    days_late.gt(0).astype("float32").where(observed_date)
)
installment_level["_IS_UNDERPAID"] = (
    installment_level["INSTALLMENT_PAID_AMOUNT"]
    .lt(installment_level["INSTALLMENT_EXPECTED_AMOUNT"] - 0.01)
    .astype("float32")
    .where(observed_amount)
)
installment_level["_EXPECTED_AMOUNT_OBSERVED"] = (
    installment_level["INSTALLMENT_EXPECTED_AMOUNT"].where(observed_amount)
)

In [13]:
# ---------- Agrégation par ancienne demande ----------
installment_grouped = installment_level.groupby(PREVIOUS_ID, observed=True)
installments_agg = installment_grouped.agg(
    INSTAL_INSTALLMENT_COUNT=(PREVIOUS_ID, "size"),
    INSTAL_LATE_PAYMENT_RATIO=("_IS_LATE", "mean"),
    INSTAL_MAX_DAYS_LATE=("_DAYS_LATE", "max"),
    INSTAL_UNDERPAYMENT_RATIO=("_IS_UNDERPAID", "mean"),
)
payment_totals = installment_grouped[
    ["INSTALLMENT_PAID_AMOUNT", "_EXPECTED_AMOUNT_OBSERVED"]
].sum(min_count=1)
expected_denominator = payment_totals["_EXPECTED_AMOUNT_OBSERVED"].mask(
    payment_totals["_EXPECTED_AMOUNT_OBSERVED"].eq(0)
)
installments_agg["INSTAL_PAYMENT_RATIO"] = (
    payment_totals["INSTALLMENT_PAID_AMOUNT"] / expected_denominator
)
installments_agg = installments_agg.reset_index()

assert installments_agg[PREVIOUS_ID].is_unique
del installment_level, installment_grouped, payment_totals
print(f"Paiements agrégés : {installments_agg.shape}")
display(installments_agg.head())

Paiements agrégés : (997752, 6)


,SK_ID_PREV,INSTAL_INSTALLMENT_COUNT,INSTAL_LATE_PAYMENT_RATIO,INSTAL_MAX_DAYS_LATE,INSTAL_UNDERPAYMENT_RATIO,INSTAL_PAYMENT_RATIO
0,1000001,2,0.0,0.0,0.0,1.0
1,1000002,4,0.0,0.0,0.0,1.0
2,1000003,3,0.0,0.0,0.0,1.0
3,1000004,7,0.0,0.0,0.0,1.0
4,1000005,10,0.2,3.0,0.0,1.0


## Réalisation de la jointure entre `installments_payments` et `previous_application`

Les caractéristiques de paiement sont désormais agrégées avec une ligne par `SK_ID_PREV`. Elles peuvent donc être jointes à `previous_application` sans multiplier les anciennes demandes. Une jointure gauche conserve toutes les demandes, y compris celles sans historique de paiement.

In [14]:
# ---------- Couverture et jointure ----------
installment_ids = pd.Index(installments_agg[PREVIOUS_ID])
previous_enriched_ids = pd.Index(previous_application_enrichi[PREVIOUS_ID])
print(
    "Identifiants de paiements absents de previous_application : "
    f"{len(installment_ids.difference(previous_enriched_ids)):,}"
)
print(
    "Anciennes demandes sans historique de paiement : "
    f"{len(previous_enriched_ids.difference(installment_ids)):,}"
)

rows_before_installment_join = len(previous_application_enrichi)
columns_before_installment_join = previous_application_enrichi.shape[1]
previous_application_enrichi = previous_application_enrichi.merge(
    installments_agg, on=PREVIOUS_ID, how="left", validate="one_to_one"
)
previous_application_enrichi["INSTAL_HAS_HISTORY"] = (
    previous_application_enrichi["INSTAL_INSTALLMENT_COUNT"]
    .notna().astype("int8")
)
previous_application_enrichi["INSTAL_INSTALLMENT_COUNT"] = (
    previous_application_enrichi["INSTAL_INSTALLMENT_COUNT"]
    .fillna(0).astype("int16")
)

assert len(previous_application_enrichi) == rows_before_installment_join
assert previous_application_enrichi[PREVIOUS_ID].is_unique

Identifiants de paiements absents de previous_application : 38,847


Anciennes demandes sans historique de paiement : 711,309


In [15]:
# ---------- Contrôle avant et après la jointure ----------
controle_installments = pd.DataFrame({
    "étape": ["avant installments", "après installments"],
    "nombre de lignes": [rows_before_installment_join, len(previous_application_enrichi)],
    "nombre de colonnes": [
        columns_before_installment_join, previous_application_enrichi.shape[1]
    ],
})
display(controle_installments)
print("Validation de la jointure installments réussie.")

,étape,nombre de lignes,nombre de colonnes
0,avant installments,1670214,42
1,après installments,1670214,48


Validation de la jointure installments réussie.


In [16]:
# ---------- Aperçu après l'étape installments ----------
display(previous_application_enrichi.head())

,SK_ID_PREV,SK_ID_CURR,NAME_CONTRACT_TYPE,AMT_ANNUITY,AMT_APPLICATION,AMT_CREDIT,AMT_DOWN_PAYMENT,AMT_GOODS_PRICE,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,...,POS_EVER_DPD,POS_EVER_SEVERE_DPD,POS_REMAINING_INSTALLMENTS_LATEST,POS_HAS_HISTORY,INSTAL_INSTALLMENT_COUNT,INSTAL_LATE_PAYMENT_RATIO,INSTAL_MAX_DAYS_LATE,INSTAL_UNDERPAYMENT_RATIO,INSTAL_PAYMENT_RATIO,INSTAL_HAS_HISTORY
0,2030495,271877,Consumer loans,1730.430,17145.0,17145.0,0.0,17145.0,SATURDAY,15,...,0.0,0.0,0.0,1,1,0.000000,0.0,0.0,1.0,1
1,2802425,108129,Cash loans,25188.615,607500.0,679671.0,NaN,607500.0,THURSDAY,11,...,0.0,0.0,32.0,1,5,0.000000,0.0,0.0,1.0,1
2,2523466,122040,Cash loans,15060.735,112500.0,136444.5,NaN,112500.0,TUESDAY,11,...,0.0,0.0,3.0,1,9,0.111111,1.0,0.0,1.0,1
3,2819243,176158,Cash loans,47041.335,450000.0,470790.0,NaN,450000.0,MONDAY,7,...,0.0,0.0,0.0,1,11,0.000000,0.0,0.0,1.0,1
4,1784265,202054,Cash loans,31924.395,337500.0,404055.0,NaN,337500.0,THURSDAY,9,...,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,0


## Logique de jointure et feature engineering entre `credit_card_balance` et `previous_application`

`credit_card_balance` contient une observation mensuelle par carte. La combinaison `SK_ID_PREV` et `MONTHS_BALANCE` étant unique, une seule agrégation par `SK_ID_PREV` est nécessaire.

Les variables retenues sont : le nombre de mois observés, le solde le plus récent, l'utilisation moyenne de la limite de crédit, le montant moyen des tirages et l'existence d'un retard sévère. Le taux d'utilisation est calculé uniquement lorsque la limite est strictement positive. `SK_DPD_DEF` est utilisé pour ignorer les petits impayés tolérés.

Les détails par type de tirage, les paiements, les créances et le statut mensuel ne sont pas retenus dans cette première version afin de limiter la redondance et le nombre de variables. `SK_ID_CURR` sert uniquement au contrôle de cohérence.

In [17]:
# ---------- Chargement de l'historique des cartes ----------
CREDIT_CARD_PATH = RAW_DIR / "credit_card_balance.csv"
if not CREDIT_CARD_PATH.is_file():
    raise FileNotFoundError(f"Fichier introuvable : {CREDIT_CARD_PATH.resolve()}")

credit_card = pd.read_csv(
    CREDIT_CARD_PATH,
    usecols=[
        "SK_ID_PREV", "SK_ID_CURR", "MONTHS_BALANCE",
        "AMT_BALANCE", "AMT_CREDIT_LIMIT_ACTUAL",
        "AMT_DRAWINGS_CURRENT", "SK_DPD_DEF",
    ],
    dtype={
        "SK_ID_PREV": "int32",
        "SK_ID_CURR": "int32",
        "MONTHS_BALANCE": "int8",
        "AMT_BALANCE": "float32",
        "AMT_CREDIT_LIMIT_ACTUAL": "float32",
        "AMT_DRAWINGS_CURRENT": "float32",
        "SK_DPD_DEF": "int16",
    },
)
print(f"credit_card_balance : {credit_card.shape}")
display(credit_card.head())

credit_card_balance : (3840312, 7)


,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,AMT_BALANCE,AMT_CREDIT_LIMIT_ACTUAL,AMT_DRAWINGS_CURRENT,SK_DPD_DEF
0,2562384,378907,-6,56.970001,135000.0,877.5,0
1,2582071,363914,-1,63975.554688,45000.0,2250.0,0
2,1740877,371185,-7,31815.224609,450000.0,0.0,0
3,1389973,337855,-4,236572.109375,225000.0,2250.0,0
4,1891521,126868,-1,453919.468750,450000.0,11547.0,0


In [18]:
# ---------- Validation des données de carte ----------
required_credit_card_columns = {
    PREVIOUS_ID, "SK_ID_CURR", MONTH_COLUMN, "AMT_BALANCE",
    "AMT_CREDIT_LIMIT_ACTUAL", "AMT_DRAWINGS_CURRENT", "SK_DPD_DEF",
}
if not required_credit_card_columns.issubset(credit_card.columns):
    missing = required_credit_card_columns.difference(credit_card.columns)
    raise KeyError(f"Colonnes absentes de credit_card_balance : {sorted(missing)}")

assert credit_card[[PREVIOUS_ID, "SK_ID_CURR", MONTH_COLUMN]].notna().all().all()
assert not credit_card.duplicated([PREVIOUS_ID, MONTH_COLUMN]).any()
assert credit_card.groupby(PREVIOUS_ID)["SK_ID_CURR"].nunique().le(1).all()
print("Validation des identifiants de carte réussie.")

Validation des identifiants de carte réussie.


In [19]:
# ---------- Feature engineering des cartes ----------
positive_credit_limit = credit_card["AMT_CREDIT_LIMIT_ACTUAL"].gt(0)
credit_card = credit_card.assign(
    _UTILIZATION=(
        credit_card["AMT_BALANCE"]
        / credit_card["AMT_CREDIT_LIMIT_ACTUAL"].where(positive_credit_limit)
    ).astype("float32"),
    _IS_SEVERE_DPD=credit_card["SK_DPD_DEF"].ge(61).astype("int8"),
)

latest_credit_card_index = credit_card.groupby(PREVIOUS_ID)[MONTH_COLUMN].idxmax()
latest_balance = (
    credit_card.loc[latest_credit_card_index, [PREVIOUS_ID, "AMT_BALANCE"]]
    .set_index(PREVIOUS_ID)
    .rename(columns={"AMT_BALANCE": "CC_BALANCE_LATEST"})
)

credit_card_agg = (
    credit_card.groupby(PREVIOUS_ID, observed=True)
    .agg(
        CC_MONTH_COUNT=(MONTH_COLUMN, "size"),
        CC_UTILIZATION_MEAN=("_UTILIZATION", "mean"),
        CC_DRAWINGS_MONTHLY_MEAN=("AMT_DRAWINGS_CURRENT", "mean"),
        CC_EVER_SEVERE_DPD=("_IS_SEVERE_DPD", "max"),
    )
    .join(latest_balance, validate="one_to_one")
    .reset_index()
)

assert credit_card_agg[PREVIOUS_ID].is_unique
del credit_card, latest_balance
print(f"Historique de carte agrégé : {credit_card_agg.shape}")
display(credit_card_agg.head())

Historique de carte agrégé : (104307, 6)


,SK_ID_PREV,CC_MONTH_COUNT,CC_UTILIZATION_MEAN,CC_DRAWINGS_MONTHLY_MEAN,CC_EVER_SEVERE_DPD,CC_BALANCE_LATEST
0,1000018,5,0.923080,29478.996094,0,136695.421875
1,1000030,8,0.630494,17257.437500,0,103027.273438
2,1000031,16,0.327365,28959.615234,0,135786.687500
3,1000035,5,0.000000,0.000000,0,0.000000
4,1000077,11,0.000000,0.000000,0,0.000000


## réalisation de la jointure entre `credit_card_balance` et `previous_application`

Les caractéristiques de carte sont désormais agrégées avec une ligne par `SK_ID_PREV`. Une jointure gauche permet de les rattacher à `previous_application` sans supprimer les demandes qui ne possèdent pas d'historique de carte et sans multiplier les lignes.

In [20]:
# ---------- Couverture et jointure ----------
credit_card_ids = pd.Index(credit_card_agg[PREVIOUS_ID])
previous_enriched_ids = pd.Index(previous_application_enrichi[PREVIOUS_ID])
print(
    "Identifiants de carte absents de previous_application : "
    f"{len(credit_card_ids.difference(previous_enriched_ids)):,}"
)
print(
    "Anciennes demandes sans historique de carte : "
    f"{len(previous_enriched_ids.difference(credit_card_ids)):,}"
)

rows_before_credit_card_join = len(previous_application_enrichi)
columns_before_credit_card_join = previous_application_enrichi.shape[1]
previous_application_enrichi = previous_application_enrichi.merge(
    credit_card_agg, on=PREVIOUS_ID, how="left", validate="one_to_one"
)
previous_application_enrichi["CC_HAS_HISTORY"] = (
    previous_application_enrichi["CC_MONTH_COUNT"].notna().astype("int8")
)
previous_application_enrichi["CC_MONTH_COUNT"] = (
    previous_application_enrichi["CC_MONTH_COUNT"].fillna(0).astype("int16")
)

assert len(previous_application_enrichi) == rows_before_credit_card_join
assert previous_application_enrichi[PREVIOUS_ID].is_unique

Identifiants de carte absents de previous_application : 11,372
Anciennes demandes sans historique de carte : 1,577,279


In [21]:
# ---------- Contrôle avant et après la jointure ----------
controle_credit_card = pd.DataFrame({
    "étape": ["avant credit_card", "après credit_card"],
    "nombre de lignes": [rows_before_credit_card_join, len(previous_application_enrichi)],
    "nombre de colonnes": [
        columns_before_credit_card_join, previous_application_enrichi.shape[1]
    ],
})
display(controle_credit_card)
print("Validation de la jointure credit_card réussie.")

,étape,nombre de lignes,nombre de colonnes
0,avant credit_card,1670214,48
1,après credit_card,1670214,54


Validation de la jointure credit_card réussie.


In [22]:
# ---------- Export et contrôle du fichier final ----------
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
previous_application_enrichi.to_csv(PREVIOUS_ENRICHI_PATH, index=False)
export_head = pd.read_csv(PREVIOUS_ENRICHI_PATH, nrows=5)
print(f"Table exportée vers : {PREVIOUS_ENRICHI_PATH.resolve()}")
display(export_head)

Table exportée vers : C:\Users\Philippe MAGNE\Documents\3 - DEV\P6_initiez_vous_au_MLOps_(partie1_sur_2)\data\processed\previous_application_enrichi.csv


,SK_ID_PREV,SK_ID_CURR,NAME_CONTRACT_TYPE,AMT_ANNUITY,AMT_APPLICATION,AMT_CREDIT,AMT_DOWN_PAYMENT,AMT_GOODS_PRICE,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,...,INSTAL_MAX_DAYS_LATE,INSTAL_UNDERPAYMENT_RATIO,INSTAL_PAYMENT_RATIO,INSTAL_HAS_HISTORY,CC_MONTH_COUNT,CC_UTILIZATION_MEAN,CC_DRAWINGS_MONTHLY_MEAN,CC_EVER_SEVERE_DPD,CC_BALANCE_LATEST,CC_HAS_HISTORY
0,2030495,271877,Consumer loans,1730.430,17145.0,17145.0,0.0,17145.0,SATURDAY,15,...,0.0,0.0,1.0,1,0,NaN,NaN,NaN,NaN,0
1,2802425,108129,Cash loans,25188.615,607500.0,679671.0,NaN,607500.0,THURSDAY,11,...,0.0,0.0,1.0,1,0,NaN,NaN,NaN,NaN,0
2,2523466,122040,Cash loans,15060.735,112500.0,136444.5,NaN,112500.0,TUESDAY,11,...,1.0,0.0,1.0,1,0,NaN,NaN,NaN,NaN,0
3,2819243,176158,Cash loans,47041.335,450000.0,470790.0,NaN,450000.0,MONDAY,7,...,0.0,0.0,1.0,1,0,NaN,NaN,NaN,NaN,0
4,1784265,202054,Cash loans,31924.395,337500.0,404055.0,NaN,337500.0,THURSDAY,9,...,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,0


## Logique d'encodage et d'agrégation de `previous_application_enrichi`

Les trois historiques sont maintenant rattachés à `previous_application`, mais la table contient encore plusieurs anciennes demandes par client. Elle doit être transformée en une table numérique possédant une ligne par `SK_ID_CURR` avant sa jointure aux datasets de modélisation.

L'encodage est volontairement ciblé. `NAME_CONTRACT_STATUS` devient des proportions de demandes approuvées et refusées ; `NAME_CONTRACT_TYPE` devient trois proportions par type de contrat ; `NAME_YIELD_GROUP` fournit uniquement la proportion de demandes approuvées à rendement élevé. Cette sélection conserve les informations métier principales sans créer plus d'une centaine de colonnes de one-hot encoding.

### Choix de traitement des colonnes natives

Les montants, annuités, durées, assurances et rendements sont calculés uniquement sur les demandes approuvées : une demande refusée ou annulée ne représente pas un crédit réellement accordé. `DAYS_DECISION` est conservée pour mesurer la récence, son maximum correspondant à la demande la plus récente puisque les valeurs sont négatives.

Les colonnes `AMT_DOWN_PAYMENT`, `RATE_DOWN_PAYMENT` et `AMT_GOODS_PRICE` sont écartées car elles sont incomplètes ou proches de `AMT_APPLICATION`. Les taux d'intérêt sont écartés avec 99,64 % de valeurs manquantes. Les cinq dates du calendrier du crédit sont écartées car elles cumulent environ 40 % de valeurs manquantes et la valeur sentinelle `365243`. Les deux indicateurs de dernière demande sont quasi constants.

Les catégories de calendrier, motif, paiement, rejet, accompagnement, client, bien, portefeuille, produit, canal, vendeur et combinaison commerciale ne sont pas encodées dans cette première version. Leur encodage complet augmenterait fortement la dimension, la mémoire et le risque de surapprentissage. Elles restent disponibles dans `previous_application_enrichi.csv` et pourront être testées individuellement après l'établissement d'un modèle de référence.

In [17]:
# ---------- Variables intermédiaires des anciennes demandes ----------
CLIENT_ID = "SK_ID_CURR"
approved = previous_application_enrichi["NAME_CONTRACT_STATUS"].eq("Approved")
application_amount_positive = previous_application_enrichi["AMT_APPLICATION"].gt(0)

previous_application_enrichi = previous_application_enrichi.assign(
    _IS_APPROVED=approved.astype("int8"),
    _IS_REFUSED=(
        previous_application_enrichi["NAME_CONTRACT_STATUS"].eq("Refused").astype("int8")
    ),
    _IS_CASH_LOAN=(
        previous_application_enrichi["NAME_CONTRACT_TYPE"].eq("Cash loans").astype("int8")
    ),
    _IS_CONSUMER_LOAN=(
        previous_application_enrichi["NAME_CONTRACT_TYPE"].eq("Consumer loans").astype("int8")
    ),
    _IS_REVOLVING_LOAN=(
        previous_application_enrichi["NAME_CONTRACT_TYPE"].eq("Revolving loans").astype("int8")
    ),
    _APPROVED_CREDIT=previous_application_enrichi["AMT_CREDIT"].where(approved),
    _APPROVED_ANNUITY=previous_application_enrichi["AMT_ANNUITY"].where(approved),
    _APPROVED_CNT_PAYMENT=previous_application_enrichi["CNT_PAYMENT"].where(approved),
    _APPROVED_CREDIT_TO_APPLICATION_RATIO=(
        previous_application_enrichi["AMT_CREDIT"]
        / previous_application_enrichi["AMT_APPLICATION"].where(application_amount_positive)
    ).where(approved),
    _INSURED_APPROVED=(
        previous_application_enrichi["NFLAG_INSURED_ON_APPROVAL"].where(approved)
    ),
    _HIGH_YIELD_APPROVED=(
        previous_application_enrichi["NAME_YIELD_GROUP"].eq("high")
        .astype("float32").where(approved)
    ),
)

In [18]:
# ---------- Agrégation des variables natives par client ----------
previous_grouped = previous_application_enrichi.groupby(CLIENT_ID, observed=True)
previous_application_par_client = previous_grouped.agg(
    PREV_APPLICATION_COUNT=(PREVIOUS_ID, "size"),
    PREV_APPROVED_RATIO=("_IS_APPROVED", "mean"),
    PREV_REFUSED_RATIO=("_IS_REFUSED", "mean"),
    PREV_DAYS_DECISION_MAX=("DAYS_DECISION", "max"),
    PREV_CASH_LOAN_RATIO=("_IS_CASH_LOAN", "mean"),
    PREV_CONSUMER_LOAN_RATIO=("_IS_CONSUMER_LOAN", "mean"),
    PREV_REVOLVING_LOAN_RATIO=("_IS_REVOLVING_LOAN", "mean"),
    PREV_APPROVED_ANNUITY_MEAN=("_APPROVED_ANNUITY", "mean"),
    PREV_APPROVED_CREDIT_TO_APPLICATION_RATIO_MEAN=(
        "_APPROVED_CREDIT_TO_APPLICATION_RATIO", "mean"
    ),
    PREV_APPROVED_CNT_PAYMENT_MEAN=("_APPROVED_CNT_PAYMENT", "mean"),
    PREV_INSURED_APPROVAL_RATIO=("_INSURED_APPROVED", "mean"),
    PREV_HIGH_YIELD_APPROVAL_RATIO=("_HIGH_YIELD_APPROVED", "mean"),
)
approved_credit_sum = (
    previous_grouped["_APPROVED_CREDIT"].sum(min_count=1)
    .rename("PREV_APPROVED_CREDIT_SUM")
)
latest_previous_index = previous_grouped["DAYS_DECISION"].idxmax()
latest_was_refused = (
    previous_application_enrichi.loc[
        latest_previous_index, [CLIENT_ID, "NAME_CONTRACT_STATUS"]
    ]
    .set_index(CLIENT_ID)["NAME_CONTRACT_STATUS"]
    .eq("Refused").astype("int8")
    .rename("PREV_LATEST_WAS_REFUSED")
)
previous_application_par_client = (
    previous_application_par_client
    .join(approved_credit_sum, validate="one_to_one")
    .join(latest_was_refused, validate="one_to_one")
)

### Agrégation des variables provenant des sous-tables

Les indicateurs de présence deviennent des proportions d'anciennes demandes couvertes. Les montants ou échéances restantes sont additionnés, les ratios sont moyennés et les indicateurs de retard sévère utilisent le maximum afin de signaler si au moins un ancien crédit est concerné.

In [19]:
# ---------- Agrégation des sous-tables par client ----------
subtable_features = previous_grouped.agg(
    PREV_POS_HISTORY_RATIO=("POS_HAS_HISTORY", "mean"),
    PREV_POS_EVER_SEVERE_DPD=("POS_EVER_SEVERE_DPD", "max"),
    PREV_INSTAL_HISTORY_RATIO=("INSTAL_HAS_HISTORY", "mean"),
    PREV_INSTAL_LATE_PAYMENT_RATIO_MEAN=("INSTAL_LATE_PAYMENT_RATIO", "mean"),
    PREV_INSTAL_MAX_DAYS_LATE=("INSTAL_MAX_DAYS_LATE", "max"),
    PREV_INSTAL_UNDERPAYMENT_RATIO_MEAN=("INSTAL_UNDERPAYMENT_RATIO", "mean"),
    PREV_INSTAL_PAYMENT_RATIO_MEAN=("INSTAL_PAYMENT_RATIO", "mean"),
    PREV_CC_HISTORY_RATIO=("CC_HAS_HISTORY", "mean"),
    PREV_CC_UTILIZATION_MEAN=("CC_UTILIZATION_MEAN", "mean"),
    PREV_CC_EVER_SEVERE_DPD=("CC_EVER_SEVERE_DPD", "max"),
)
subtable_sums = (
    previous_grouped[["POS_REMAINING_INSTALLMENTS_LATEST", "CC_BALANCE_LATEST"]]
    .sum(min_count=1)
    .rename(columns={
        "POS_REMAINING_INSTALLMENTS_LATEST": "PREV_POS_REMAINING_INSTALLMENTS_SUM",
        "CC_BALANCE_LATEST": "PREV_CC_BALANCE_LATEST_SUM",
    })
)
previous_application_par_client = (
    previous_application_par_client
    .join(subtable_features, validate="one_to_one")
    .join(subtable_sums, validate="one_to_one")
    .reset_index()
)

assert previous_application_par_client[CLIENT_ID].is_unique
assert previous_application_par_client.select_dtypes(exclude="number").empty
print(
    f"previous_application par client : {previous_application_par_client.shape}"
)
display(previous_application_par_client.head())

KeyError: "Label(s) ['CC_EVER_SEVERE_DPD', 'CC_HAS_HISTORY', 'CC_UTILIZATION_MEAN'] do not exist"

In [ ]:
# ---------- Export de previous_application encodé ----------
PREVIOUS_CLIENT_ENCODED_PATH = (
    PROCESSED_DIR / "previous_application_par_client_encoded.csv"
)
previous_application_par_client.to_csv(PREVIOUS_CLIENT_ENCODED_PATH, index=False)
previous_client_export_head = pd.read_csv(PREVIOUS_CLIENT_ENCODED_PATH, nrows=5)
print(f"Table exportée vers : {PREVIOUS_CLIENT_ENCODED_PATH.resolve()}")
display(previous_client_export_head)

# Libération de la table détaillée avant le chargement des applications
del previous_application_enrichi, previous_grouped, subtable_features, subtable_sums

Table exportée vers : C:\Users\Philippe MAGNE\Documents\3 - DEV\P6_initiez_vous_au_MLOps_(partie1_sur_2)\data\processed\previous_application_par_client_encoded.csv


,SK_ID_CURR,PREV_APPLICATION_COUNT,PREV_APPROVED_RATIO,PREV_REFUSED_RATIO,PREV_DAYS_DECISION_MAX,PREV_CASH_LOAN_RATIO,PREV_CONSUMER_LOAN_RATIO,PREV_REVOLVING_LOAN_RATIO,PREV_APPROVED_ANNUITY_MEAN,PREV_APPROVED_CREDIT_TO_APPLICATION_RATIO_MEAN,...,PREV_INSTAL_HISTORY_RATIO,PREV_INSTAL_LATE_PAYMENT_RATIO_MEAN,PREV_INSTAL_MAX_DAYS_LATE,PREV_INSTAL_UNDERPAYMENT_RATIO_MEAN,PREV_INSTAL_PAYMENT_RATIO_MEAN,PREV_CC_HISTORY_RATIO,PREV_CC_UTILIZATION_MEAN,PREV_CC_EVER_SEVERE_DPD,PREV_POS_REMAINING_INSTALLMENTS_SUM,PREV_CC_BALANCE_LATEST_SUM
0,100001,1,1.0,0.0,-1740,0.000000,1.000000,0.0,3951.000,0.957782,...,1.0,0.000000,0.0,0.0,1.0,0.0,NaN,NaN,0.0,NaN
1,100002,1,1.0,0.0,-606,0.000000,1.000000,0.0,9251.775,1.000000,...,1.0,0.000000,0.0,0.0,1.0,0.0,NaN,NaN,6.0,NaN
2,100003,3,1.0,0.0,-746,0.333333,0.666667,0.0,56553.990,1.057664,...,1.0,0.000000,0.0,0.0,1.0,0.0,NaN,NaN,1.0,NaN
3,100004,1,1.0,0.0,-815,0.000000,1.000000,0.0,5357.250,0.828021,...,1.0,0.000000,0.0,0.0,1.0,0.0,NaN,NaN,0.0,NaN
4,100005,2,0.5,0.0,-315,0.500000,0.500000,0.0,4813.200,0.899950,...,0.5,0.111111,1.0,0.0,1.0,0.0,NaN,NaN,0.0,NaN


## Réalisation des jointures avec les applications enrichies par `bureau`

La table des anciennes demandes possède désormais une ligne par `SK_ID_CURR`. Elle est jointe séparément au train et au test avec une jointure gauche et une validation `one_to_one`. `PREV_HAS_HISTORY` distingue les clients sans ancienne demande connue. Les deux jeux restent séparés afin d'éviter toute confusion ou fuite lors de la modélisation.

In [27]:
# ---------- Chargement des applications enrichies par bureau ----------
TRAIN_BUREAU_PATH = PROCESSED_DIR / "application_train_enrichi_bureau_encoded.csv"
TEST_BUREAU_PATH = PROCESSED_DIR / "application_test_enrichi_bureau_encoded.csv"
TRAIN_FINAL_PATH = PROCESSED_DIR / "application_train_final_encoded.csv"
TEST_FINAL_PATH = PROCESSED_DIR / "application_test_final_encoded.csv"

for data_path in [TRAIN_BUREAU_PATH, TEST_BUREAU_PATH]:
    if not data_path.is_file():
        raise FileNotFoundError(f"Fichier introuvable : {data_path.resolve()}")

application_train_bureau = pd.read_csv(TRAIN_BUREAU_PATH)
application_test_bureau = pd.read_csv(TEST_BUREAU_PATH)
assert application_train_bureau[CLIENT_ID].is_unique
assert application_test_bureau[CLIENT_ID].is_unique

In [28]:
# ---------- Jointures séparées du train et du test ----------
application_train_final = application_train_bureau.merge(
    previous_application_par_client,
    on=CLIENT_ID, how="left", validate="one_to_one",
)
application_test_final = application_test_bureau.merge(
    previous_application_par_client,
    on=CLIENT_ID, how="left", validate="one_to_one",
)

train_previous_indicator = (
    application_train_final["PREV_APPLICATION_COUNT"]
    .notna().astype("int8").rename("PREV_HAS_HISTORY")
)
test_previous_indicator = (
    application_test_final["PREV_APPLICATION_COUNT"]
    .notna().astype("int8").rename("PREV_HAS_HISTORY")
)
application_train_final = pd.concat(
    [application_train_final, train_previous_indicator], axis=1
)
application_test_final = pd.concat(
    [application_test_final, test_previous_indicator], axis=1
)
application_train_final["PREV_APPLICATION_COUNT"] = (
    application_train_final["PREV_APPLICATION_COUNT"].fillna(0).astype("int16")
)
application_test_final["PREV_APPLICATION_COUNT"] = (
    application_test_final["PREV_APPLICATION_COUNT"].fillna(0).astype("int16")
)

In [29]:
# ---------- Validation des jointures finales ----------
assert len(application_train_final) == len(application_train_bureau)
assert len(application_test_final) == len(application_test_bureau)
assert application_train_final[CLIENT_ID].is_unique
assert application_test_final[CLIENT_ID].is_unique
assert "TARGET" in application_train_final.columns
assert "TARGET" not in application_test_final.columns
assert (
    application_train_final.drop(columns="TARGET").columns.tolist()
    == application_test_final.columns.tolist()
), "Les variables explicatives du train et du test ne sont pas alignées."

controle_final = pd.DataFrame({
    "jeu de données": ["train", "test"],
    "lignes avant": [len(application_train_bureau), len(application_test_bureau)],
    "lignes après": [len(application_train_final), len(application_test_final)],
    "colonnes avant": [
        application_train_bureau.shape[1], application_test_bureau.shape[1]
    ],
    "colonnes après": [
        application_train_final.shape[1], application_test_final.shape[1]
    ],
})
controle_final["écart de lignes"] = (
    controle_final["lignes après"] - controle_final["lignes avant"]
)
display(controle_final)
print("Toutes les validations finales sont réussies.")

,jeu de données,lignes avant,lignes après,colonnes avant,colonnes après,écart de lignes
0,train,307510,307510,294,321,0
1,test,48744,48744,293,320,0


Toutes les validations finales sont réussies.


In [30]:
# ---------- Export et aperçu des datasets finaux ----------
application_train_final.to_csv(TRAIN_FINAL_PATH, index=False)
application_test_final.to_csv(TEST_FINAL_PATH, index=False)
train_final_head = pd.read_csv(TRAIN_FINAL_PATH, nrows=5)
test_final_head = pd.read_csv(TEST_FINAL_PATH, nrows=5)

print(f"Train exporté vers : {TRAIN_FINAL_PATH.resolve()}")
display(train_final_head)
print(f"Test exporté vers : {TEST_FINAL_PATH.resolve()}")
display(test_final_head)

Train exporté vers : C:\Users\Philippe MAGNE\Documents\3 - DEV\P6_initiez_vous_au_MLOps_(partie1_sur_2)\data\processed\application_train_final_encoded.csv


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,...,PREV_INSTAL_LATE_PAYMENT_RATIO_MEAN,PREV_INSTAL_MAX_DAYS_LATE,PREV_INSTAL_UNDERPAYMENT_RATIO_MEAN,PREV_INSTAL_PAYMENT_RATIO_MEAN,PREV_CC_HISTORY_RATIO,PREV_CC_UTILIZATION_MEAN,PREV_CC_EVER_SEVERE_DPD,PREV_POS_REMAINING_INSTALLMENTS_SUM,PREV_CC_BALANCE_LATEST_SUM,PREV_HAS_HISTORY
0,100002,1,0,0,1,0,202500.0,406597.5,24700.5,351000.0,...,0.0,0.0,0.0,1.0,0.000000,NaN,NaN,6.0,NaN,1
1,100003,0,0,0,0,0,270000.0,1293502.5,35698.5,1129500.0,...,0.0,0.0,0.0,1.0,0.000000,NaN,NaN,1.0,NaN,1
2,100004,0,1,1,1,0,67500.0,135000.0,6750.0,135000.0,...,0.0,0.0,0.0,1.0,0.000000,NaN,NaN,0.0,NaN,1
3,100006,0,0,0,1,0,135000.0,312682.5,29686.5,297000.0,...,0.0,0.0,0.0,1.0,0.111111,0.0,0.0,3.0,0.0,1
4,100007,0,0,0,1,0,121500.0,513000.0,21865.5,513000.0,...,0.3,12.0,0.0,1.0,0.000000,NaN,NaN,14.0,NaN,1


Test exporté vers : C:\Users\Philippe MAGNE\Documents\3 - DEV\P6_initiez_vous_au_MLOps_(partie1_sur_2)\data\processed\application_test_final_encoded.csv


,SK_ID_CURR,NAME_CONTRACT_TYPE,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,REGION_POPULATION_RELATIVE,...,PREV_INSTAL_LATE_PAYMENT_RATIO_MEAN,PREV_INSTAL_MAX_DAYS_LATE,PREV_INSTAL_UNDERPAYMENT_RATIO_MEAN,PREV_INSTAL_PAYMENT_RATIO_MEAN,PREV_CC_HISTORY_RATIO,PREV_CC_UTILIZATION_MEAN,PREV_CC_EVER_SEVERE_DPD,PREV_POS_REMAINING_INSTALLMENTS_SUM,PREV_CC_BALANCE_LATEST_SUM,PREV_HAS_HISTORY
0,100001,0,0,1,0,135000.0,568800.0,20560.5,450000.0,0.018850,...,0.000000,0.0,0.0,1.0,0.0,NaN,NaN,0.0,NaN,1
1,100005,0,0,1,0,99000.0,222768.0,17370.0,180000.0,0.035792,...,0.111111,1.0,0.0,1.0,0.0,NaN,NaN,0.0,NaN,1
2,100013,0,1,1,0,202500.0,663264.0,69777.0,630000.0,0.019101,...,0.267974,21.0,0.0,1.0,0.0,NaN,NaN,0.0,NaN,1
3,100028,0,0,1,2,315000.0,1575000.0,49018.5,1575000.0,0.026392,...,0.049550,7.0,0.0,1.0,0.2,0.035934,0.0,0.0,37335.914,1
4,100038,0,1,0,1,180000.0,625500.0,32067.0,625500.0,0.010032,...,0.000000,0.0,0.0,1.0,0.0,NaN,NaN,0.0,NaN,1


## Conclusion

`previous_application_enrichi` a été encodé de manière ciblée et agrégé au niveau client avant deux jointures séparées avec les applications enrichies par `bureau`. Les datasets finaux conservent une ligne par `SK_ID_CURR`, possèdent les mêmes variables explicatives et sont prêts pour les contrôles précédant la modélisation.